# Aerial OBB Object Detection & Benchmark Suite (Windows Anaconda Runner)
### Comprehensive Jupyter Notebook for Local Execution on Windows Anaconda

This notebook is engineered specifically for **Windows Anaconda / Jupyter Notebook** environments to eliminate common Windows-specific pitfalls:
1. **Multiple Initialization Prevention**: Sets `KMP_DUPLICATE_LIB_OK=TRUE` to resolve the `libiomp5md.dll` crash when multiple OpenMP runtimes (e.g. MKL/Torch/Ultralytics) are imported.
2. **Windows Console & Character Encoding Safety**: Reconfigures standard streams to UTF-8 with automatic fallback replacing problematic Unicode characters (such as tick marks `✓`, degree symbols `°`, and delta symbols `Δ`) that trigger `UnicodeEncodeError: charmap codec cannot encode character` on Windows `cp1252` consoles.
3. **Windows Multiprocessing Safety**: Disables PyTorch multi-worker multiprocessing spawn crashes in Jupyter by defaulting to single-process DataLoader (`workers=0` or safe main guards).
4. **Local Directory Management**: Replaces Google Drive mounts with self-contained local paths (`./data`, `./weights`, `./results`) with automatic extraction of bundled annotations (`assets/*_annotations.zip`).
5. **All 10 Survey Architectures Supported**: `std`, `rvsa`, `ars-detr`, `oriented-former`, `rio-detr`, `rhino`, `ao2-detr`, `swin-obb`, `lsknet`, `yolo11-obb` across `visdrone`, `codrone`, and `dota` datasets.


In [ ]:
# Step 1: Windows Anaconda Environment Setup & Crash Mitigation
# Resolves OpenMP double-initialization (libiomp5md.dll) and Windows-1252/cp1252 encoding crashes
import os
import sys
import io

# 1. Prevent OpenMP libiomp5md.dll multiple initialization error
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Force UTF-8 encoding across Windows console streams
os.environ["PYTHONIOENCODING"] = "utf-8"
if sys.platform == "win32":
    if hasattr(sys.stdout, "reconfigure"):
        try:
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
            sys.stderr.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass
    else:
        try:
            sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
            sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="replace")
        except Exception:
            pass

# 3. Add Project Root to sys.path
from pathlib import Path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("[OK] Windows Anaconda environment configured successfully.")
print(f"[*] Python Executable : {sys.executable}")
print(f"[*] Platform          : {sys.platform}")
print(f"[*] Working Directory : {project_root}")

In [ ]:
# Step 2: Hardware Acceleration & PyTorch CUDA Check
import torch

print("=== Hardware Acceleration Status ===")
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available : {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device     : {device_name} (x{device_count})")
    print(f"Total VRAM     : {vram_gb:.2f} GB")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    preferred_device = "cuda"
else:
    print("[!] No NVIDIA CUDA GPU detected or CUDA runtime not loaded. Using CPU mode.")
    print("    Tip: If you have an NVIDIA GPU, install PyTorch with CUDA via:")
    print("    conda install pytorch torchvision pytorch-cuda=12.1 -c pytorch -c nvidia")
    preferred_device = "cpu"

print(f"[OK] Preferred Device: '{preferred_device}'")

In [ ]:
# Step 3: Install / Verify Python Dependencies in Anaconda
# Runs install.py or direct pip requirements
import subprocess
import sys

print("[*] Verifying dependencies from requirements.txt...")
cmd = [sys.executable, "install.py"]
res = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(res.stdout)
if res.returncode != 0:
    print(f"[!] Notice during dependency installation:
{res.stderr}")
else:
    print("[OK] All dependencies validated.")

In [ ]:
# Step 4: Local Directory Structure Setup
# Sets up local directories for data, weights, and benchmark results
from pathlib import Path

DATA_DIR = Path("data")
WEIGHTS_DIR = Path("weights")
RESULTS_DIR = Path("results")
ASSETS_DIR = Path("assets")

for folder in [DATA_DIR, WEIGHTS_DIR, RESULTS_DIR, ASSETS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("[OK] Local workspace directories initialized:")
print(f"  - Datasets : {DATA_DIR.resolve()}")
print(f"  - Weights  : {WEIGHTS_DIR.resolve()}")
print(f"  - Results  : {RESULTS_DIR.resolve()}")
print(f"  - Assets   : {ASSETS_DIR.resolve()}")

In [ ]:
# Step 5: Annotation Extraction & Dataset Readiness Verification
# Automatically unpacks bundled annotations from assets/ to data/ if present
import zipfile
from pathlib import Path
from src.data.downloader import verify_dataset_status

datasets = ["visdrone", "codrone", "dota"]
print("[*] Checking datasets and unpacking bundled annotations if available...")

for d_name in datasets:
    zpath = Path(f"assets/{d_name}_annotations.zip")
    target_d = Path(f"data/{d_name}")
    if zpath.exists():
        target_d.mkdir(parents=True, exist_ok=True)
        has_txt = any(target_d.rglob("*.txt"))
        has_xml = any(target_d.rglob("*.xml"))
        if not has_txt and not has_xml:
            print(f"[*] Unpacking bundled annotations for {d_name} from {zpath.name}...")
            with zipfile.ZipFile(zpath, "r") as z:
                z.extractall(target_d)
            print(f"[OK] Annotations unpacked for {d_name}.")

print("
--- DATASET STATUS CHECK ---")
for d in datasets:
    status = verify_dataset_status(d, Path("data"))
    if status["ready"]:
        print(f"  [{d.upper():<8}] [READY] Found {status['num_images']} images across splits: {status['splits_found']}")
    else:
        print(f"  [{d.upper():<8}] [PENDING] {status['num_images']} images found. Pass --download in pipeline or add images to data/{d}")

In [ ]:
# Step 6: Quick Smoke Test (Verifies complete pipeline with synthetic batches in ~30s)
# Note: --train-workers 0 is specified to avoid Windows multiprocessing spawn overhead/errors in Jupyter.
!python scripts/run_pipeline.py \n    --datasets visdrone \n    --models std rvsa ars-detr oriented-former rio-detr rhino ao2-detr swin-obb lsknet yolo11-obb \n    --mode full \n    --test \n    --train-workers 0 \n    --device auto \n    --save-plots \n    --run-name smoke_test_windows

In [ ]:
# Step 7: Configurable Production Pipeline Runner
# Choose models, datasets, epochs, and sample limits suitable for your Windows machine
# Tip: Set --train-workers 0 for Windows Jupyter compatibility.
# Tip: Use --val-samples 100 for fast empirical benchmarking or omit for full validation split.

!python scripts/run_pipeline.py \n    --steps pre_eval train post_eval report \n    --models yolo11-obb lsknet std rvsa ars-detr oriented-former rio-detr rhino ao2-detr swin-obb \n    --datasets visdrone codrone dota \n    --epochs 5 \n    --val-samples 100 \n    --train-workers 0 \n    --device auto \n    --save-plots \n    --resume True \n    --download \n    --run-name local_windows_run

In [ ]:
# Step 8: Verification of Generated Checkpoints & Benchmark CSVs
import glob
from pathlib import Path
import pandas as pd

print("=== 1. SAVED MODEL WEIGHTS (.pt) ===")
weights_dir = Path("weights")
found_weights = list(weights_dir.rglob("*.pt")) if weights_dir.exists() else []
if found_weights:
    for f in sorted(found_weights):
        print(f"  [OK] Checkpoint: {f.name} ({f.stat().st_size / 1e6:.2f} MB) in {f.parent}")
else:
    print("  [*] No .pt files found yet in weights/")

print("
=== 2. BENCHMARK SUMMARY METRICS (CSV) ===")
csv_files = sorted(glob.glob("results/**/benchmark_summary.csv", recursive=True))
if csv_files:
    latest_csv = csv_files[-1]
    print(f"Displaying latest results: {latest_csv}
")
    df = pd.read_csv(latest_csv)
    cols = [c for c in ["model", "dataset", "phase", "map50", "map75", "f1", "angle_mae", "fps"] if c in df.columns]
    print(df[cols].to_string(index=False))
else:
    print("  [*] No benchmark_summary.csv found yet in results/")

In [ ]:
# Step 9: Inline Visualization & Markdown Report Viewer in Jupyter
import os
import glob
from pathlib import Path
from IPython.display import Image, display, Markdown

run_dirs = [p for p in glob.glob("results/*") if os.path.isdir(p) and not p.endswith(".tmp")]
run_dirs = sorted(run_dirs, key=os.path.getmtime)

if run_dirs:
    latest_run = run_dirs[-1]
    print(f"Latest Run Directory: {latest_run}
")
    
    # 1. Display Markdown Benchmark Report
    report_file = os.path.join(latest_run, "benchmark_report.md")
    if os.path.exists(report_file):
        with open(report_file, encoding="utf-8", errors="replace") as f:
            display(Markdown(f.read()))
            
    # 2. Display Pre- vs Post-Training Performance Gain Chart
    pre_post_chart = os.path.join(latest_run, "plots", "pre_vs_post_comparison.png")
    if os.path.exists(pre_post_chart):
        print("
=======================================================")
        print("--- Pre-Training vs Post-Training Performance Gain ---")
        print("=======================================================")
        display(Image(filename=pre_post_chart))
        
    # 3. Display Overall Model Benchmark Comparison Chart
    comp_chart = os.path.join(latest_run, "plots", "model_benchmark_comparison.png")
    if os.path.exists(comp_chart):
        print("
--- Overall Model Benchmark Comparison ---")
        display(Image(filename=comp_chart))
        
    # 4. Display Confusion Matrices and Regression Plots
    for img_path in sorted(glob.glob(f"{latest_run}/plots/*.png")):
        bname = os.path.basename(img_path)
        if bname not in ("model_benchmark_comparison.png", "pre_vs_post_comparison.png"):
            print(f"
Displaying: {bname}")
            display(Image(filename=img_path))
else:
    print("[*] No results found yet. Run Step 6 or 7 first.")